# GTSRB Research: Experiment 4 — Concept Bottleneck Model (CBM) ResNet-18

This notebook implements **Experiment 4 (Representation-Centric Robustness)** from our controlled research ladder:
$$X \to \text{ResNet-18 Backbone} \to \text{Concept Bottleneck } \hat{C} \in \mathbb{R}^{22} \to \text{Class Head } \hat{Y} \in \mathbb{R}^{43}$$

### Methodological Grounding:
- **Koh et al. (ICML 2020):** *Concept Bottleneck Models*.
- Uses the legally grounded **Vienna Convention / German StVO** concept ontology across 4 orthogonal axes:
  1. **Shape** (6 classes: circle, triangle_up, triangle_down, octagon, diamond, rectangle)
  2. **Colour Profile** (5 classes: white_red, blue_white, yellow_white, white_black, red_white)
  3. **Border Type** (4 classes: red_border, white_border, black_border, none)
  4. **Symbol Category** (7 classes: speed_numeral, warning_hazard, mandatory_direction, prohibition_restriction, priority_geometry, deregulation_cancel, none)
  Total Bottleneck Dimension = $6 + 5 + 4 + 7 = 22$.
- Multi-task objective: $\mathcal{L} = \mathcal{L}_{task}(y, \hat{y}) + \lambda_c \sum_k \mathcal{L}_k(c_k, \hat{c}_k) + \lambda_{aux} \mathcal{L}_{aux}(y, \hat{y}_{aux})$.

### Experimental Controls (Strictly Identical to Experiments 1, 2, & 3):
1. **Backbone:** ResNet-18 with ImageNet initialization and Zero-$\gamma$ residual BN initialization.
2. **Optimization:** SGD (Nesterov $\mu=0.9$, lr=0.01, weight decay $10^{-4}$ with No-Bias-Decay rule).
3. **Schedule:** 3-epoch linear warmup + Cosine Annealing decay over 15 epochs.
4. **Evaluation:** Clean accuracy, Macro-F1, ECE, concept accuracies, and 15 held-out corruptions $\times$ 5 severities ($mCA$ and $RPD$).

In [ ]:
# --- Step 1: Environment Setup & GPU Verification ---
import os
from google.colab import drive
drive.mount('/content/drive')

# Check if dataset is already unzipped from prior runs
if not os.path.exists('/content/ML_Project/german'):
    print('Extracting base dataset from Drive...')
    !cp "/content/drive/MyDrive/ML_Project_Data.zip" /content/
    !unzip -q /content/ML_Project_Data.zip -d /content/ML_Project
else:
    print('Base dataset already present at /content/ML_Project!')

%cd /content/ML_Project

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: Architecture & Concept Bottleneck Sanity Check ---
import torch
from src.models.cbm import ConceptBottleneckModel, CBMLoss, get_concept_lookup_tensors
from src.models.concept_encoder import BOTTLENECK_DIM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConceptBottleneckModel(num_classes=43, pretrained=False, bottleneck_mode='residual').to(device)
model.eval()

# Create dummy batch of 4 images
dummy_x = torch.randn(4, 3, 64, 64, device=device)
with torch.no_grad():
    out = model(dummy_x)

print(f"Input Shape            : {dummy_x.shape}")
print(f"Class Logits Shape     : {out['logits'].shape}  (Expected: [4, 43])")
print(f"Concept Bottleneck (C) : {out['c_hat'].shape}   (Expected: [4, {BOTTLENECK_DIM}])")
print(f"Shape Logits           : {out['logits_shape'].shape}   (Expected: [4, 6])")
print(f"Colour Logits          : {out['logits_colour'].shape}   (Expected: [4, 5])")
print(f"Border Logits          : {out['logits_border'].shape}   (Expected: [4, 4])")
print(f"Symcat Logits          : {out['logits_symcat'].shape}   (Expected: [4, 7])")
print(f"Visual Features        : {out['features'].shape} (Expected: [4, 512])")

# Verify GPU concept lookup tensors
shape_lut, colour_lut, border_lut, symcat_lut = get_concept_lookup_tensors(device)
print(f"Lookup tensors created on {device}: Shape={shape_lut.shape}, Colour={colour_lut.shape}")
print("Concept Bottleneck Model Architecture Verified Successfully!")

In [ ]:
# --- Step 3: Train Concept Bottleneck Model (Colab T4 GPU) ---
# 15 epochs takes ~4-6 minutes on T4 with Mixed Precision (AMP)
!python train_cbm.py \
    --data-root german \
    --manifest-dir data/master_v1/metadata \
    --output-dir checkpoints/cbm_resnet18 \
    --seed 42 \
    --epochs 15 \
    --warmup-epochs 3 \
    --batch-size 64 \
    --lr 0.01 \
    --lambda-c 1.0 \
    --lambda-aux 0.5 \
    --bottleneck-mode residual

In [ ]:
# --- Step 4: Plot Multi-Task Loss and Concept Validation Curves ---
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('checkpoints/cbm_resnet18/training_history.json', 'r') as f:
    history = json.load(f)

epochs_df = pd.DataFrame(history['epochs'])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 1. Multi-Task Losses
ax1.plot(epochs_df['epoch'], epochs_df['train_loss'], label='Total Train Loss', color='black', lw=2)
ax1.plot(epochs_df['epoch'], epochs_df['train_loss_task'], label='Task Loss (Class)', color='blue', lw=1.5, ls='--')
ax1.plot(epochs_df['epoch'], epochs_df['train_loss_concepts'], label='Concept Loss (Sum)', color='red', lw=1.5, ls='--')
ax1.plot(epochs_df['epoch'], epochs_df['val_loss'], label='Val Loss', color='purple', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Multi-Task Loss Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Sign Classification Performance
ax2.plot(epochs_df['epoch'], epochs_df['val_acc'] * 100, label='Val Accuracy (%)', color='green', lw=2)
ax2.plot(epochs_df['epoch'], epochs_df['val_macro_f1'] * 100, label='Val Macro-F1 (%)', color='navy', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.set_title('Sign Classification Performance')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Concept Head Accuracies
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_shape'] * 100, label='Shape Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_colour'] * 100, label='Colour Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_border'] * 100, label='Border Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_symcat'] * 100, label='Symbol Cat Acc (%)', lw=1.8)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Concept Accuracy (%)')
ax3.set_title('Per-Axis Concept Accuracy')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Clean Test & Robustness Benchmark (15 Corruptions x 5 Severities) ---
!python evaluate_robustness.py \
    --checkpoint checkpoints/cbm_resnet18/best_cbm_resnet18.pt \
    --data-root german \
    --meta-path data/master_v1/metadata/test.csv \
    --output-dir reports/cbm_resnet18 \
    --batch-size 128

In [ ]:
# --- Step 6: 4-Way Comparative Analysis: Baseline vs. AugMix vs. Frequency vs. CBM ---
import json
from pathlib import Path
import pandas as pd

p_base = Path('reports/baseline_resnet18/evaluation_summary.json')
p_aug = Path('reports/augmix_resnet18/evaluation_summary.json')
p_freq = Path('reports/frequency_resnet18/evaluation_summary.json')
p_cbm = Path('reports/cbm_resnet18/evaluation_summary.json')

summaries = {}
for name, p in [
    ('Baseline ResNet-18', p_base),
    ('AugMix ResNet-18', p_aug),
    ('Frequency ResNet-18', p_freq),
    ('CBM ResNet-18', p_cbm),
]:
    if p.exists():
        with open(p) as f:
            summaries[name] = json.load(f)

if 'CBM ResNet-18' in summaries:
    metrics = [
        ('Clean Accuracy', 'clean_accuracy', True),
        ('Clean Macro-F1', 'clean_macro_f1', True),
        ('Expected Calibration Error (ECE)', 'clean_ece', False),
        ('Mean Corruption Accuracy (mCA)', 'mean_corruption_accuracy', True),
        ('Relative Performance Drop (RPD)', 'relative_performance_drop', False),
    ]
    comp_records = []
    for title, key, is_pct in metrics:
        rec = {'Metric': title}
        for model_name, s in summaries.items():
            val = s.get(key, 0.0)
            rec[model_name] = f"{val*100:.2f}%" if is_pct else f"{val:.4f}"
        comp_records.append(rec)
    display(pd.DataFrame(comp_records))

    # Save backup to Google Drive
    !mkdir -p "/content/drive/MyDrive/ML_Project_Results/cbm_resnet18"
    !cp -r checkpoints/cbm_resnet18/* "/content/drive/MyDrive/ML_Project_Results/cbm_resnet18/"
    !cp -r reports/cbm_resnet18/* "/content/drive/MyDrive/ML_Project_Results/cbm_resnet18/"
    print("\nAll CBM Experiment 4 results backed up to Google Drive successfully!")
else:
    print("CBM evaluation summary not found yet. Complete Step 5 first.")